# Cross-check: the pipeline vs pandas and DuckDB SQL

The analyses in `src/` are written in the Python standard library, so the whole
pipeline reruns anywhere with no install and CI can prove it is byte-stable.

This notebook recomputes the headline numbers **with the standard analyst
toolkit** and asserts they match what the pipeline wrote to `analysis/`.
If any `assert` fails, the notebook (and CI) fails.

| Result | Pipeline (`src/`) | Recomputed here with |
| --- | --- | --- |
| Historical CLV by country | `crm_retention.py` | pandas `groupby` **and** DuckDB SQL |
| RFM segments | `crm_retention.py` | pandas `rank` + `np.select` |
| Cohort retention | `crm_retention.py` | pandas `pivot_table` |
| Time to second order | `crm_retention.py` | pandas `cumcount` |

In [1]:
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
crm = json.loads((ROOT / "analysis/crm_retention_metrics.json").read_text())
orders = pd.read_csv(ROOT / "data/online_retail_orders.csv", dtype={"customer_id": str},
                     parse_dates=["order_date"])
print(f"{len(orders):,} real orders, {orders.customer_id.nunique():,} customers (GBP)")

36,969 real orders, 5,878 customers (GBP)


## 1. Historical CLV by country — pandas

In [2]:
cust = orders.groupby("customer_id").agg(
    country=("country", "first"), n_orders=("order_id", "count"),
    monetary=("order_value_gbp", "sum"), last=("order_date", "max"))

clv = (cust.groupby("country")
           .agg(customers=("n_orders", "size"), orders=("n_orders", "sum"), revenue=("monetary", "sum"))
           .query("customers >= 30"))
clv["historical_clv_gbp"] = clv.revenue / clv.customers
clv = clv.sort_values("historical_clv_gbp", ascending=False).round(2)

expected = pd.DataFrame(crm["clv_by_country"]["ranked"]).set_index("country")
assert list(clv.index) == list(expected.index)
assert np.allclose(clv.historical_clv_gbp, expected.historical_clv_gbp, atol=0.01)
clv[["customers", "historical_clv_gbp"]]

,customers,historical_clv_gbp
country,,
Germany,107,3985.97
France,93,3734.29
Spain,39,2762.45
United Kingdom,5349,2700.27


## 2. The same number in SQL — DuckDB

In [3]:
sql = f"""
WITH per_customer AS (
    SELECT customer_id, any_value(country) AS country, SUM(order_value_gbp) AS monetary
    FROM read_csv_auto('{(ROOT / "data/online_retail_orders.csv").as_posix()}', types={{'customer_id': 'VARCHAR'}})
    GROUP BY customer_id
)
SELECT country, COUNT(*) AS customers, ROUND(SUM(monetary) / COUNT(*), 2) AS historical_clv_gbp
FROM per_customer
GROUP BY country
HAVING COUNT(*) >= 30
ORDER BY historical_clv_gbp DESC
"""
by_sql = duckdb.sql(sql).df().set_index("country")
assert np.allclose(by_sql.historical_clv_gbp, expected.historical_clv_gbp, atol=0.01)
by_sql

,customers,historical_clv_gbp
country,,
Germany,107,3985.97
France,93,3734.29
Spain,39,2762.45
United Kingdom,5349,2700.27


## 3. RFM segments — pandas

Quintile scores from the average rank (ties share one score), then the same
segment rules as `crm_retention._segment`.

In [4]:
as_of = orders.order_date.max() + pd.Timedelta(days=1)
n = len(cust)

def score(values, higher_is_better=True):
    pos = values.rank(method="average", ascending=not higher_is_better) - 1  # 0-based midrank
    return 5 - np.minimum(4, np.floor(pos * 5 / n)).astype(int)

r = score((as_of - cust["last"]).dt.days, higher_is_better=False)
# np.round rounds half to even (2.5 -> 2, 3.5 -> 4), exactly like Python's round() in the pipeline
fm = np.round((score(cust.n_orders) + score(cust.monetary)) / 2).astype(int)
segment = np.select(
    [(r >= 4) & (fm >= 4), (r >= 3) & (fm >= 3), (r >= 4) & (fm <= 2),
     (r == 3) & (fm <= 3), r == 2, (r <= 1) & (fm >= 4)],
    ["Champions", "Loyal customers", "New / promising", "Needs attention", "At risk", "Can't lose them"],
    default="Hibernating")
counts = pd.Series(segment, index=cust.index).value_counts()

expected_seg = {s["segment"]: s["customers"] for s in crm["rfm"]["segments"]}
assert counts.to_dict() == expected_seg, (counts.to_dict(), expected_seg)
counts.rename("customers").to_frame()

,customers
Champions,1537
At risk,1174
Hibernating,1059
Loyal customers,937
New / promising,581
Needs attention,471
Can't lose them,119


## 4. Cohort retention — pandas `pivot_table`

In [5]:
orders["cohort"] = orders.cohort_month
orders["offset"] = ((orders.order_date.dt.year - orders.cohort.str[:4].astype(int)) * 12
                    + orders.order_date.dt.month - orders.cohort.str[5:].astype(int))
active = (orders[orders.offset.between(0, 6)]
          .pivot_table(index="cohort", columns="offset", values="customer_id", aggfunc="nunique", fill_value=0))
retention = active.div(orders.groupby("cohort").customer_id.nunique(), axis=0).round(4)

# A month the extract does not fully cover is unknown, not 0% (right-censoring).
last = orders.order_date.max()
last_full = last.to_period("M") - (0 if last.is_month_end else 1)
cohorts = pd.PeriodIndex(retention.index, freq="M")
for k in retention.columns[1:]:
    retention.loc[cohorts + k > last_full, k] = np.nan

expected_ret = pd.DataFrame({c["cohort_month"]: c["retention"] for c in crm["cohort_retention"]["cohorts"]}).T
assert np.allclose(retention.values, expected_ret.values.astype(float), atol=1e-4, equal_nan=True)
(retention.head(6) * 100).round(1)  # % of the cohort active in month k

offset,0,1,2,3,4,5,6
cohort,,,,,,,
2009-12,100.0,35.3,33.4,42.5,38.0,35.9,37.7
2010-01,100.0,20.6,31.1,30.6,26.4,30.0,25.8
2010-02,100.0,23.8,22.5,29.1,24.6,20.0,19.2
2010-03,100.0,19.0,23.0,24.2,23.2,20.3,24.6
2010-04,100.0,19.4,19.4,16.3,18.4,22.4,27.6
2010-05,100.0,15.8,16.9,17.3,17.7,25.6,21.3


## 5. Time to second order - pandas

Days from the first to the second order, counting a second invoice on the same day as the
same visit. Only customers with a full year of data after their first order, and not the
first month (it also holds customers acquired before the data starts).

In [6]:
days = (orders[["customer_id", "order_date"]].drop_duplicates()
        .sort_values(["customer_id", "order_date"]))
days["n"] = days.groupby("customer_id").cumcount()
first = days[days.n == 0].set_index("customer_id").order_date
second = days[days.n == 1].set_index("customer_id").order_date
gap = (second.reindex(first.index) - first).dt.days

data_end = orders.order_date.max() + pd.Timedelta(days=1)
keep = ((first.dt.to_period("M") != orders.order_date.min().to_period("M"))
        & ((data_end - first).dt.days >= 365))
gap = gap[keep]

t2 = crm["time_to_second_order"]
assert len(gap) == t2["customers"]
for day, share in t2["share_back_by_day"].items():
    assert abs((gap <= int(day)).mean() - share) < 1e-4, day
assert gap[gap <= 365].median() == t2["median_days_if_back"]
pd.Series({f"by day {d}": round((gap <= d).mean() * 100, 1) for d in (30, 60, 90, 180, 365)},
          name="% of new customers who ordered again")

by day 30     16.4
by day 60     31.0
by day 90     40.8
by day 180    57.1
by day 365    69.4
Name: % of new customers who ordered again, dtype: float64

**All assertions passed:** the dependency-free pipeline and the standard toolkit agree on every headline number.